# PRÁCTICA 4 — NOTEBOOK 1: CALIDAD DE DATOS
**Dataset:** Delitos Informáticos en Colombia  
**Objetivo:** Perfilado, diagnóstico y limpieza de datos  
**Variable objetivo:** `CRIMINALIDAD` (SI / NO)

## 0. Instalación e importación de librerías

## Estructura de directorios

Se crearán las siguientes carpetas para organizar los outputs:
- `outputs/reportes/` — Reportes HTML
- `outputs/datasets/` — Datasets CSV limpios
- `outputs/imagenes/` — Gráficos PNG


In [ ]:
import os

# Crear estructura de directorios para outputs
output_dirs = {
    'reportes': 'outputs/reportes',
    'datasets': 'outputs/datasets',
    'imagenes': 'outputs/imagenes'
}

for dir_name, dir_path in output_dirs.items():
    os.makedirs(dir_path, exist_ok=True)
    print(f'✓ Carpeta creada/verificada: {dir_path}')

print('\nEstructura de directorios lista.')


In [1]:
# Instalar dependencias correctamente
import sys
import subprocess

# Limpiar caché de pip y reinstalar desde cero
subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], capture_output=True)

# Instalar ydata-profiling desde PyPI con todas las dependencias
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "ydata-profiling", "--no-cache-dir"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("ydata-profiling instalado correctamente")
else:
    print("Advertencia:", result.stderr[:200])


ydata-profiling instalado correctamente


In [4]:
import sys
!{sys.executable} -m pip install --upgrade pip setuptools wheel

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Workaround completo para ydata-profiling + pkg_resources
import sys
from types import SimpleNamespace

# Crear un mock de pkg_resources que sea compatible con ydata-profiling
class MockDistribution:
    def __init__(self, version):
        self.version = version

class MockPkgResources:
    @staticmethod
    def get_distribution(package_name):
        """Simula pkg_resources.get_distribution()"""
        try:
            from importlib.metadata import version
            return MockDistribution(version(package_name))
        except:
            return MockDistribution("unknown")

# Reemplazar pkg_resources en sys.modules ANTES de importar ydata_profiling
sys.modules['pkg_resources'] = MockPkgResources()

from ydata_profiling import ProfileReport

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

print('Librerías importadas correctamente')

Librerías importadas correctamente


## Estructura de directorios

Se crearán las siguientes carpetas para organizar los outputs:
- `outputs/reportes/` — Reportes HTML  
- `outputs/datasets/` — Datasets CSV limpios
- `outputs/imagenes/` — Gráficos PNG


In [24]:
import os

# Crear estructura de directorios para outputs
output_dirs = {
    'reportes': 'outputs/reportes',
    'datasets': 'outputs/datasets',
    'imagenes': 'outputs/imagenes'
}

for dir_name, dir_path in output_dirs.items():
    os.makedirs(dir_path, exist_ok=True)
    print(f'✓ Carpeta creada/verificada: {dir_path}')

print('\nEstructura de directorios lista.')


✓ Carpeta creada/verificada: outputs/reportes
✓ Carpeta creada/verificada: outputs/datasets
✓ Carpeta creada/verificada: outputs/imagenes

Estructura de directorios lista.


---
## 1. CARGA DE DATOS ORIGINALES

In [3]:
# Cargar el dataset original (con tildes, sin limpiar)
df_original = pd.read_csv('Delitos_Informaticos_V1_20260216.csv', low_memory=False)

print(f'Dimensiones del dataset: {df_original.shape[0]:,} filas × {df_original.shape[1]} columnas')
print(f'Columnas: {list(df_original.columns)}')
df_original.head()

Dimensiones del dataset: 63,705 filas × 17 columnas
Columnas: ['CRIMINALIDAD', 'ES_ARCHIVO', 'ES_PRECLUSIÓN', 'ESTADO', 'ETAPA_CASO', 'LEY', 'PAÍS_HECHO', 'DEPARTAMENTO_HECHO', 'MUNICIPIO_HECHO', 'SECCIONAL', 'AÑO_HECHOS', 'AÑO_ENTRADA', 'AÑO_DENUNCIA', 'DELITO', 'GRUPO_DELITO', 'CONSUMADO', 'TOTAL_PROCESOS']


,CRIMINALIDAD,ES_ARCHIVO,ES_PRECLUSIÓN,ESTADO,ETAPA_CASO,LEY,PAÍS_HECHO,DEPARTAMENTO_HECHO,MUNICIPIO_HECHO,SECCIONAL,AÑO_HECHOS,AÑO_ENTRADA,AÑO_DENUNCIA,DELITO,GRUPO_DELITO,CONSUMADO,TOTAL_PROCESOS
0,NO,SI,NO,INACTIVO,INDAGACIÓN,Ley 906,Colombia,Santander,FLORIDABLANCA,DIRECCIÓN SECCIONAL DE SANTANDER,2020,2020,2020,VIOLACION DE DATOS PERSONALES ART 269F LEY 1273 DE 2009,DELITOS INFORMATICOS,NO APLICA,50
1,NO,SI,NO,INACTIVO,INDAGACIÓN,Ley 906,Colombia,"BOGOTÁ, D. C.","BOGOTÁ, D.C.",DIRECCIÓN SECCIONAL DE BOGOTÁ,2023,2023,2023,"VIOLACION DE DATOS PERSONALES ART 269F LEY 1273 DE 2009,...",DELITOS INFORMATICOS,NO APLICA,123
2,SI,NO,NO,ACTIVO,INDAGACIÓN,Ley 906,Colombia,Caquetá,FLORENCIA,DIRECCIÓN SECCIONAL DE CAQUETÁ,2025,2025,2025,HURTO POR MEDIOS INFORMATICOS Y SEMEJANTES ART. 269I LEY...,DELITOS INFORMATICOS,NO APLICA,104
3,SI,SI,NO,INACTIVO,INDAGACIÓN,Ley 906,Colombia,Antioquia,ITAGUI,DIRECCIÓN SECCIONAL DE MEDELLÍN,2021,2021,2021,ACCESO ABUSIVO A UN SISTEMA INFORMATICO ART 269A LEY 127...,DELITOS INFORMATICOS,NO APLICA,31
4,SI,NO,NO,ACTIVO,INDAGACIÓN,Ley 906,Colombia,Cundinamarca,CHÍA,DIRECCIÓN SECCIONAL DE CUNDINAMARCA,2025,2025,2025,ACCESO ABUSIVO A UN SISTEMA INFORMATICO ART 269A LEY 127...,DELITOS INFORMATICOS,NO APLICA,92


---
## 2. PERFILADO DE DATOS (ydata-profiling)

Se genera un reporte automático HTML con estadísticas descriptivas, distribuciones, correlaciones, valores únicos y alertas de calidad para cada columna del dataset original.

In [ ]:
# Generar el perfil de datos con ydata-profiling
print('Generando perfil de datos... Por favor espera...')

profile = ProfileReport(
    df_original,
    title='Perfil de Calidad de Datos — Delitos Informáticos Colombia',
    explorative=True,
    minimal=False
)

# Guardar el reporte en HTML
reporte_path = 'outputs/reportes/reporte_perfilado_datos.html'
profile.to_file(reporte_path)
print(f'Reporte HTML guardado como: {reporte_path}')

Generando perfil de datos... Por favor espera...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 102.96it/s]

Reporte HTML guardado como: reporte_perfilado_datos.html


---
## 3. DIAGNÓSTICO DE CALIDAD DE DATOS

Se evalúan las **5 dimensiones de calidad de datos** basándonos en los resultados del perfilado.

### 3.1 Vista general del dataset

In [5]:
print('='*60)
print('RESUMEN GENERAL DEL DATASET')
print('='*60)
print(f'Filas totales:     {df_original.shape[0]:,}')
print(f'Columnas totales:  {df_original.shape[1]}')
print(f'Duplicados:        {df_original.duplicated().sum():,}')
print(f'Nulos totales:     {df_original.isnull().sum().sum():,}')
print()
print('Tipos de datos:')
print(df_original.dtypes)

RESUMEN GENERAL DEL DATASET
Filas totales:     63,705
Columnas totales:  17
Duplicados:        0
Nulos totales:     0

Tipos de datos:
CRIMINALIDAD          object
ES_ARCHIVO            object
ES_PRECLUSIÓN         object
ESTADO                object
ETAPA_CASO            object
LEY                   object
PAÍS_HECHO            object
DEPARTAMENTO_HECHO    object
MUNICIPIO_HECHO       object
SECCIONAL             object
AÑO_HECHOS             int64
AÑO_ENTRADA            int64
AÑO_DENUNCIA          object
DELITO                object
GRUPO_DELITO          object
CONSUMADO             object
TOTAL_PROCESOS         int64
dtype: object


### 3.2 Dimensión 1 — COMPLETITUD
> ¿Existen valores nulos o faltantes en el dataset?

In [7]:
nulos = df_original.isnull().sum()
pct_nulos = (nulos / len(df_original)) * 100

completitud = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': pct_nulos.round(2),
    'Completitud (%)': (100 - pct_nulos).round(2)
})

print('DIMENSIÓN 1 — COMPLETITUD')
print(completitud.to_string())
print()

completitud_global = 100 - (df_original.isnull().sum().sum() / (df_original.shape[0] * df_original.shape[1]) * 100)
print(f'Completitud global del dataset: {completitud_global:.2f}%')

if completitud_global == 100:
    print('El dataset NO presenta valores nulos. Completitud perfecta.')
else:
    print(f'Hay datos faltantes en algunas columnas. Se requiere imputación.')

DIMENSIÓN 1 — COMPLETITUD
                    Nulos  Porcentaje (%)  Completitud (%)
CRIMINALIDAD            0             0.0            100.0
ES_ARCHIVO              0             0.0            100.0
ES_PRECLUSIÓN           0             0.0            100.0
ESTADO                  0             0.0            100.0
ETAPA_CASO              0             0.0            100.0
LEY                     0             0.0            100.0
PAÍS_HECHO              0             0.0            100.0
DEPARTAMENTO_HECHO      0             0.0            100.0
MUNICIPIO_HECHO         0             0.0            100.0
SECCIONAL               0             0.0            100.0
AÑO_HECHOS              0             0.0            100.0
AÑO_ENTRADA             0             0.0            100.0
AÑO_DENUNCIA            0             0.0            100.0
DELITO                  0             0.0            100.0
GRUPO_DELITO            0             0.0            100.0
CONSUMADO               0     

In [ ]:
# Gráfico de completitud
fig, ax = plt.subplots(figsize=(10, 5))
completitud['Completitud (%)'].sort_values().plot(
    kind='barh', ax=ax, color='steelblue', edgecolor='white'
)
ax.axvline(x=100, color='green', linestyle='--', label='100% completo')
ax.set_title('Completitud por columna (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Completitud (%)')
ax.set_xlim(0, 105)
for i, v in enumerate(completitud['Completitud (%)'].sort_values()):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=8)
plt.tight_layout()
grafico_path = 'outputs/imagenes/completitud.png'
plt.savefig(grafico_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Gráfico guardado como: {grafico_path}')

Gráfico guardado como completitud.png


### 3.3 Dimensión 2 — UNICIDAD
> ¿Existen registros duplicados en el dataset?

In [10]:
duplicados = df_original.duplicated().sum()
pct_dup = (duplicados / len(df_original)) * 100

print('DIMENSIÓN 2 — UNICIDAD')
print(f'Registros totales:     {len(df_original):,}')
print(f'Registros duplicados:  {duplicados:,} ({pct_dup:.2f}%)')
print(f'Registros únicos:      {len(df_original) - duplicados:,}')
print()

# Columnas con baja cardinalidad (muchos valores repetidos)
print('Valores únicos por columna (cardinalidad):')
for col in df_original.columns:
    nunique = df_original[col].nunique()
    pct = nunique / len(df_original) * 100
    flag = 'MUY BAJA CARDINALIDAD' if nunique == 1 else ('Baja' if pct < 0.1 else '')
    print(f'  {col:<25} → {nunique:>5} únicos ({pct:.3f}%) {flag}')

print()
if duplicados == 0:
    print('No se detectaron filas duplicadas.')
else:
    print(f'Se encontraron {duplicados:,} registros duplicados. Deben eliminarse.')

DIMENSIÓN 2 — UNICIDAD
Registros totales:     63,705
Registros duplicados:  0 (0.00%)
Registros únicos:      63,705

Valores únicos por columna (cardinalidad):
  CRIMINALIDAD              →     2 únicos (0.003%) Baja
  ES_ARCHIVO                →     2 únicos (0.003%) Baja
  ES_PRECLUSIÓN             →     2 únicos (0.003%) Baja
  ESTADO                    →     2 únicos (0.003%) Baja
  ETAPA_CASO                →     5 únicos (0.008%) Baja
  LEY                       →     3 únicos (0.005%) Baja
  PAÍS_HECHO                →     2 únicos (0.003%) Baja
  DEPARTAMENTO_HECHO        →    34 únicos (0.053%) Baja
  MUNICIPIO_HECHO           →  1024 únicos (1.607%) 
  SECCIONAL                 →    52 únicos (0.082%) Baja
  AÑO_HECHOS                →    17 únicos (0.027%) Baja
  AÑO_ENTRADA               →    17 únicos (0.027%) Baja
  AÑO_DENUNCIA              →    18 únicos (0.028%) Baja
  DELITO                    →    67 únicos (0.105%) 
  GRUPO_DELITO              →     1 únicos (0.002%

### 3.4 Dimensión 3 — CONSISTENCIA
> ¿Los datos son coherentes internamente? ¿Hay inconsistencias lógicas entre columnas?

In [11]:
print('DIMENSIÓN 3 — CONSISTENCIA')
print()

# Verificación 1: Columnas con un solo valor único (sin variabilidad)
print('--- Columnas con un solo valor único (sin variabilidad informativa) ---')
for col in df_original.columns:
    if df_original[col].nunique() == 1:
        val = df_original[col].unique()[0]
        print(f'{col}: solo contiene "{val}" en {len(df_original):,} registros → columna constante')

# Verificación 2: Año de hechos vs año de entrada
print()
print('--- Consistencia temporal: AÑO_HECHOS vs AÑO_ENTRADA ---')
df_original['AÑO_HECHOS_num'] = pd.to_numeric(df_original['AÑO_HECHOS'], errors='coerce')
df_original['AÑO_ENTRADA_num'] = pd.to_numeric(df_original['AÑO_ENTRADA'], errors='coerce')
inconsistentes = df_original[df_original['AÑO_HECHOS_num'] > df_original['AÑO_ENTRADA_num']]
print(f'  Registros donde AÑO_HECHOS > AÑO_ENTRADA: {len(inconsistentes):,}')
if len(inconsistentes) > 0:
    print('Inconsistencia lógica: el hecho ocurre DESPUÉS del ingreso al sistema')
else:
    print('Consistencia temporal correcta')

# Limpieza de columnas auxiliares
df_original.drop(columns=['AÑO_HECHOS_num','AÑO_ENTRADA_num'], inplace=True)

# Verificación 3: Tildes y codificación en nombres de columnas
print()
print('--- Caracteres especiales en nombres de columnas ---')
cols_con_tildes = [c for c in df_original.columns if any(ch in c for ch in 'ÁÉÍÓÚÑÜ')]
if cols_con_tildes:
    print(f'Columnas con tildes/caracteres especiales: {cols_con_tildes}')
    print('Estos nombres pueden causar errores en sistemas externos y deben normalizarse.')
else:
    print('No se encontraron caracteres especiales en nombres de columnas.')

DIMENSIÓN 3 — CONSISTENCIA

--- Columnas con un solo valor único (sin variabilidad informativa) ---
GRUPO_DELITO: solo contiene "DELITOS INFORMATICOS" en 63,705 registros → columna constante
CONSUMADO: solo contiene "NO APLICA" en 63,705 registros → columna constante

--- Consistencia temporal: AÑO_HECHOS vs AÑO_ENTRADA ---
  Registros donde AÑO_HECHOS > AÑO_ENTRADA: 0
Consistencia temporal correcta

--- Caracteres especiales en nombres de columnas ---
Columnas con tildes/caracteres especiales: ['ES_PRECLUSIÓN', 'PAÍS_HECHO', 'AÑO_HECHOS', 'AÑO_ENTRADA', 'AÑO_DENUNCIA']
Estos nombres pueden causar errores en sistemas externos y deben normalizarse.


### 3.5 Dimensión 4 — VALIDEZ
> ¿Los valores están dentro de rangos y formatos esperados?

In [12]:
print('DIMENSIÓN 4 — VALIDEZ')
print()

# Validar años (rango esperado: 2000 - año actual)
print('--- Rango de años ---')
for col_anio in ['AÑO_HECHOS', 'AÑO_ENTRADA']:
    serie = pd.to_numeric(df_original[col_anio], errors='coerce')
    fuera_rango = ((serie < 2000) | (serie > 2026)).sum()
    print(f'  {col_anio}: min={serie.min()}, max={serie.max()}, '
          f'fuera de rango [2000-2026]: {fuera_rango}')
    if fuera_rango == 0:
        print(f'Valores válidos')
    else:
        print(f'{fuera_rango} registros fuera del rango esperado')

# Validar columnas binarias (SI/NO)
print()
print('--- Columnas binarias (valores esperados: SI / NO) ---')
binarias = ['CRIMINALIDAD', 'ES_ARCHIVO', 'ES_PRECLUSIÓN']
for col in binarias:
    vals = df_original[col].unique()
    invalidos = [v for v in vals if v not in ['SI', 'NO']]
    print(f'  {col}: valores encontrados → {list(vals)}')
    if invalidos:
        print(f'Valores inválidos: {invalidos}')
    else:
        print(f'Valores válidos')

# Validar TOTAL_PROCESOS (debe ser > 0)
print()
print('--- TOTAL_PROCESOS (debe ser > 0) ---')
negativos = (df_original['TOTAL_PROCESOS'] <= 0).sum()
print(f'  Registros con TOTAL_PROCESOS <= 0: {negativos}')
if negativos == 0:
    print('Todos los valores son positivos')
else:
    print(f'{negativos} registros con valores inválidos')

DIMENSIÓN 4 — VALIDEZ

--- Rango de años ---
  AÑO_HECHOS: min=2010, max=2026, fuera de rango [2000-2026]: 0
Valores válidos
  AÑO_ENTRADA: min=2010, max=2026, fuera de rango [2000-2026]: 0
Valores válidos

--- Columnas binarias (valores esperados: SI / NO) ---
  CRIMINALIDAD: valores encontrados → ['NO', 'SI']
Valores válidos
  ES_ARCHIVO: valores encontrados → ['SI', 'NO']
Valores válidos
  ES_PRECLUSIÓN: valores encontrados → ['NO', 'SI']
Valores válidos

--- TOTAL_PROCESOS (debe ser > 0) ---
  Registros con TOTAL_PROCESOS <= 0: 0
Todos los valores son positivos


### 3.6 Dimensión 5 — EXACTITUD
> ¿Los datos representan correctamente la realidad? ¿Existen valores atípicos (outliers)?

In [13]:
print('DIMENSIÓN 5 — EXACTITUD')
print()

# Análisis de outliers en TOTAL_PROCESOS (única columna numérica relevante)
col = 'TOTAL_PROCESOS'
serie = df_original[col]
Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR
outliers = ((serie < lim_inf) | (serie > lim_sup)).sum()

print(f'Columna: {col}')
print(f'  Media:         {serie.mean():.2f}')
print(f'  Mediana:       {serie.median():.2f}')
print(f'  Desv. típica:  {serie.std():.2f}')
print(f'  Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}')
print(f'  Límite inf.:   {lim_inf:.2f}')
print(f'  Límite sup.:   {lim_sup:.2f}')
print(f'  Outliers (IQR): {outliers:,} ({outliers/len(serie)*100:.2f}%)')
print(f'  Valor máximo:  {serie.max():,}')

# Verificar valores extraños en PAIS_HECHO
print()
print(f'País hecho - valores únicos: {df_original["PAÍS_HECHO"].unique()}')
sin_dato_pais = (df_original['PAÍS_HECHO'] == 'SIN DATO').sum()
print(f'  Registros con "SIN DATO" en PAÍS_HECHO: {sin_dato_pais:,} ({sin_dato_pais/len(df_original)*100:.2f}%)')

DIMENSIÓN 5 — EXACTITUD

Columna: TOTAL_PROCESOS
  Media:         8.40
  Mediana:       1.00
  Desv. típica:  104.38
  Q1=1.0, Q3=3.0, IQR=2.0
  Límite inf.:   -2.00
  Límite sup.:   6.00
  Outliers (IQR): 7,502 (11.78%)
  Valor máximo:  14,425

País hecho - valores únicos: ['Colombia' 'SIN DATO']
  Registros con "SIN DATO" en PAÍS_HECHO: 17 (0.03%)


In [ ]:
# Visualización de outliers
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot
axes[0].boxplot(df_original['TOTAL_PROCESOS'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[0].set_title('Distribución TOTAL_PROCESOS\n(con outliers)', fontweight='bold')
axes[0].set_ylabel('Total de Procesos')

# Histograma
axes[1].hist(df_original['TOTAL_PROCESOS'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(lim_sup, color='red', linestyle='--', label=f'Límite sup. IQR = {lim_sup:.0f}')
axes[1].set_title('Histograma TOTAL_PROCESOS', fontweight='bold')
axes[1].set_xlabel('Total de Procesos')
axes[1].legend()

plt.tight_layout()
grafico_path = 'outputs/imagenes/outliers_total_procesos.png'
plt.savefig(grafico_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Gráfico guardado como: {grafico_path}')

Gráfico guardado como outliers_total_procesos.png


---
## 4. LIMPIEZA Y MEJORA DE DATOS

Se implementan todos los pasos de limpieza identificados en el diagnóstico.

### 4.1 Paso 1 — Eliminar columnas constantes (sin información)

In [16]:
df_limpio = df_original.copy()

# Eliminar columnas con un solo valor único (no aportan al modelo)
cols_constantes = [col for col in df_limpio.columns if df_limpio[col].nunique() == 1]
print(f'Columnas constantes a eliminar: {cols_constantes}')
df_limpio.drop(columns=cols_constantes, inplace=True)
print(f'Eliminadas {len(cols_constantes)} columna(s) constante(s)')
print(f'Dataset: {df_limpio.shape[0]:,} filas × {df_limpio.shape[1]} columnas')

Columnas constantes a eliminar: ['GRUPO_DELITO', 'CONSUMADO']
Eliminadas 2 columna(s) constante(s)
Dataset: 63,705 filas × 15 columnas


### 4.2 Paso 2 — Normalizar nombres de columnas (quitar tildes)

In [17]:
import unicodedata

def normalizar_nombre(nombre):
    """Elimina tildes y caracteres especiales de nombres de columnas."""
    nfkd = unicodedata.normalize('NFKD', nombre)
    sin_tildes = ''.join(c for c in nfkd if not unicodedata.combining(c))
    return sin_tildes.replace(' ', '_').upper()

cols_antes = list(df_limpio.columns)
df_limpio.columns = [normalizar_nombre(c) for c in df_limpio.columns]
cols_despues = list(df_limpio.columns)

print('Renombrado de columnas:')
for a, d in zip(cols_antes, cols_despues):
    cambio = ' → ' + d if a != d else '(sin cambio)'
    print(f'  {a}{cambio}')

Renombrado de columnas:
  CRIMINALIDAD(sin cambio)
  ES_ARCHIVO(sin cambio)
  ES_PRECLUSIÓN → ES_PRECLUSION
  ESTADO(sin cambio)
  ETAPA_CASO(sin cambio)
  LEY(sin cambio)
  PAÍS_HECHO → PAIS_HECHO
  DEPARTAMENTO_HECHO(sin cambio)
  MUNICIPIO_HECHO(sin cambio)
  SECCIONAL(sin cambio)
  AÑO_HECHOS → ANO_HECHOS
  AÑO_ENTRADA → ANO_ENTRADA
  AÑO_DENUNCIA → ANO_DENUNCIA
  DELITO(sin cambio)
  TOTAL_PROCESOS(sin cambio)


### 4.3 Paso 3 — Estandarizar tipos de datos

In [18]:
# Convertir años a entero
for col in ['ANO_HECHOS', 'ANO_ENTRADA']:
    if col in df_limpio.columns:
        df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce').astype('Int64')
        print(f'{col} convertido a Int64')

# Convertir ANO_DENUNCIA a entero
if 'ANO_DENUNCIA' in df_limpio.columns:
    df_limpio['ANO_DENUNCIA'] = pd.to_numeric(df_limpio['ANO_DENUNCIA'], errors='coerce').astype('Int64')
    print('ANO_DENUNCIA convertido a Int64')

# Estandarizar columnas de texto a mayúsculas
cols_texto = df_limpio.select_dtypes(include='object').columns
for col in cols_texto:
    df_limpio[col] = df_limpio[col].str.strip().str.upper()
print(f'{len(cols_texto)} columnas de texto estandarizadas a mayúsculas')

ANO_HECHOS convertido a Int64
ANO_ENTRADA convertido a Int64
ANO_DENUNCIA convertido a Int64
11 columnas de texto estandarizadas a mayúsculas


### 4.4 Paso 4 — Tratar outliers en TOTAL_PROCESOS

In [19]:
# Aplicar capping al percentil 99 (winsorización)
p99 = df_limpio['TOTAL_PROCESOS'].quantile(0.99)
n_outliers = (df_limpio['TOTAL_PROCESOS'] > p99).sum()

print(f'Límite de capping (percentil 99): {p99:.0f}')
print(f'Registros con valores superiores: {n_outliers:,}')

df_limpio['TOTAL_PROCESOS'] = df_limpio['TOTAL_PROCESOS'].clip(upper=p99)

print(f'Outliers tratados con capping (winsorización al P99)')
print(f'Nuevo máximo: {df_limpio["TOTAL_PROCESOS"].max()}')

Límite de capping (percentil 99): 116
Registros con valores superiores: 636
Outliers tratados con capping (winsorización al P99)
Nuevo máximo: 116


### 4.5 Paso 5 — Verificar duplicados y limpiarlos

In [20]:
n_antes = len(df_limpio)
df_limpio.drop_duplicates(inplace=True)
n_despues = len(df_limpio)

print(f'Registros antes de eliminar duplicados: {n_antes:,}')
print(f'Registros después:                      {n_despues:,}')
print(f'Registros eliminados:                   {n_antes - n_despues:,}')
print(f'Paso completado')

Registros antes de eliminar duplicados: 63,705
Registros después:                      63,705
Registros eliminados:                   0
Paso completado


### 4.6 Paso 6 — Crear variable DECADA a partir de AÑO_HECHOS (feature engineering)

In [21]:
# Crear una variable categórica por período de tiempo
def clasificar_periodo(ano):
    if pd.isna(ano):
        return 'DESCONOCIDO'
    elif ano <= 2015:
        return 'ANTES_2015'
    elif ano <= 2019:
        return '2016_2019'
    elif ano <= 2022:
        return '2020_2022'
    else:
        return '2023_ADELANTE'

df_limpio['PERIODO_HECHO'] = df_limpio['ANO_HECHOS'].apply(clasificar_periodo)
print('Variable PERIODO_HECHO creada:')
print(df_limpio['PERIODO_HECHO'].value_counts())

Variable PERIODO_HECHO creada:
PERIODO_HECHO
2023_ADELANTE    22547
2020_2022        19236
2016_2019        12855
ANTES_2015        9067
Name: count, dtype: int64


### 4.7 Resumen final del dataset limpio

In [ ]:
# Gráfico de distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución de CRIMINALIDAD
vc = df_limpio['CRIMINALIDAD'].value_counts()
colors = ['#2196F3', '#FF5722']
axes[0].pie(vc, labels=vc.index, autopct='%1.1f%%', colors=colors,
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Distribución Variable Objetivo\nCRIMINALIDAD', fontweight='bold')

# Distribución por departamento (top 10)
top10_dpto = df_limpio['DEPARTAMENTO_HECHO'].value_counts().head(10)
top10_dpto.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Top 10 Departamentos con más\nDelitos Informáticos', fontweight='bold')
axes[1].set_xlabel('Cantidad de registros')
axes[1].invert_yaxis()

plt.tight_layout()
grafico_path = 'outputs/imagenes/distribucion_variables.png'
plt.savefig(grafico_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Gráfico guardado como: {grafico_path}')

✅ Gráfico guardado como distribucion_variables.png


### 4.8 Guardar dataset limpio

In [ ]:
# Guardar el dataset limpio para usarlo en el Notebook 2
dataset_path = 'outputs/datasets/Delitos_Informaticos_Limpio.csv'
df_limpio.to_csv(dataset_path, index=False, encoding='utf-8')

print(f'Dataset limpio guardado como: {dataset_path}')
print(f'Dimensiones finales: {df_limpio.shape[0]:,} filas × {df_limpio.shape[1]} columnas')
print()
print('Primeras filas del dataset limpio:')
df_limpio.head()

Dataset limpio guardado como: Delitos_Informaticos_Limpio.csv
   Dimensiones finales: 63,705 filas × 16 columnas

Primeras filas del dataset limpio:


,CRIMINALIDAD,ES_ARCHIVO,ES_PRECLUSION,ESTADO,ETAPA_CASO,LEY,PAIS_HECHO,DEPARTAMENTO_HECHO,MUNICIPIO_HECHO,SECCIONAL,ANO_HECHOS,ANO_ENTRADA,ANO_DENUNCIA,DELITO,TOTAL_PROCESOS,PERIODO_HECHO
0,NO,SI,NO,INACTIVO,INDAGACIÓN,LEY 906,COLOMBIA,SANTANDER,FLORIDABLANCA,DIRECCIÓN SECCIONAL DE SANTANDER,2020,2020,2020,VIOLACION DE DATOS PERSONALES ART 269F LEY 1273 DE 2009,50,2020_2022
1,NO,SI,NO,INACTIVO,INDAGACIÓN,LEY 906,COLOMBIA,"BOGOTÁ, D. C.","BOGOTÁ, D.C.",DIRECCIÓN SECCIONAL DE BOGOTÁ,2023,2023,2023,"VIOLACION DE DATOS PERSONALES ART 269F LEY 1273 DE 2009,...",116,2023_ADELANTE
2,SI,NO,NO,ACTIVO,INDAGACIÓN,LEY 906,COLOMBIA,CAQUETÁ,FLORENCIA,DIRECCIÓN SECCIONAL DE CAQUETÁ,2025,2025,2025,HURTO POR MEDIOS INFORMATICOS Y SEMEJANTES ART. 269I LEY...,104,2023_ADELANTE
3,SI,SI,NO,INACTIVO,INDAGACIÓN,LEY 906,COLOMBIA,ANTIOQUIA,ITAGUI,DIRECCIÓN SECCIONAL DE MEDELLÍN,2021,2021,2021,ACCESO ABUSIVO A UN SISTEMA INFORMATICO ART 269A LEY 127...,31,2020_2022
4,SI,NO,NO,ACTIVO,INDAGACIÓN,LEY 906,COLOMBIA,CUNDINAMARCA,CHÍA,DIRECCIÓN SECCIONAL DE CUNDINAMARCA,2025,2025,2025,ACCESO ABUSIVO A UN SISTEMA INFORMATICO ART 269A LEY 127...,92,2023_ADELANTE
